# 03 · Report and backup

Builds the same report as the Databricks job (`index.html`, `report.xlsx`, per-pair CSVs,
`summary.json`) into `<run>/report/`, and copies protocol/report files to
`s3://.../runs/<run_id>/` in the distributed layout. Local `_local/` data is never uploaded.

In [ ]:
# Parameters: keep CONFIG / OUT identical in notebooks 00-03
CONFIG = "c.json"  # pairs config (col_map files resolve next to it)
OUT = ".dtrack-local/runs"  # local runs root; one folder per run id
RUN_ID = None  # None: 00 starts a new run, 01-03 continue the run 00 opened (OUT/LATEST)
INLINE = False  # True when presigned downloads are blocked (<= 25 MiB per window)
PAIRS = None  # e.g. ["orders"]; None = every active pair
RUN_S3 = None  # None: DTRACK_RUN_S3 (stable runs root), or no backup

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 40, "display.width", 200)

from dtrack_local import pipeline

In [ ]:
RUN_ID = RUN_ID or (Path(OUT) / "LATEST").read_text().strip()
run = pipeline.open_run(
    CONFIG, run_id=RUN_ID, out=OUT, inline=INLINE, pairs=PAIRS
)
print("run_id:", run.run_id)
print("folder:", run.folder.resolve())
print("status:", run.status())

RUN_S3 = RUN_S3 or os.getenv("DTRACK_RUN_S3")
print("runs root:", RUN_S3)

In [ ]:
built = pipeline.report(run)
summary = built["summary"]
print(
    json.dumps(
        {
            k: summary[k]
            for k in ("run_id", "status", "findings", "units_expected", "units_completed")
        },
        indent=2,
    )
)
pd.Series(built["files"])

Open `report/index.html` in a browser; `report.xlsx` holds the same tables.

In [ ]:
if RUN_S3:
    saved = pipeline.backup(run, RUN_S3)
    print(json.dumps(saved, indent=2))
else:
    print("RUN_S3 not set: nothing uploaded")
print("status:", run.status())